# **Animals10 Image Classification — ML Pipeline**
### CNN-based Image Classification Coursework

---
## Notebook Objectives
Build, train and test a Convolutional Neural Network (CNN) for an image classification task using the **Animals10** dataset from Kaggle.

The pipeline covers:
1. Data collection, validation and preparation
2. Exploratory Data Analysis (EDA)
3. Building and training a CNN model
4. Model evaluation and hyperparameter optimisation
5. Prediction on unseen data and individual input

---
## Inputs
* Animals10 dataset downloaded from Kaggle: `https://www.kaggle.com/datasets/alessiocorrado99/animals10/data`
* Expected folder structure after download and extraction:
```
datasets/
  raw-img/
    cane/        (dog)
    cavallo/     (horse)
    elefante/    (elephant)
    farfalla/    (butterfly)
    gallina/     (chicken)
    gatto/       (cat)
    mucca/       (cow)
    pecora/      (sheep)
    ragno/       (spider)
    scoiattolo/  (squirrel)
```

---
## Outputs
* Pre-processed dataset: `../datasets/animals10_processed.npz`
* Trained model: `../models/animals10_cnn_v1.h5`

---
## Notes / Comments
* The dataset folder names are in Italian. A translation map is provided in the notebook.
* Images vary in size and are resized to 64×64 pixels for this pipeline.
* AI tools were not used in the preparation of this coursework.

---
---
---
## Step 1 — Import Libraries

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
import itertools
import random
import PIL
from PIL import Image

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import joblib

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Show error messages only
sns.set_style('whitegrid')

I0000 00:00:1774881430.320969    4174 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774881430.342198    4174 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774881451.948939    4174 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774881463.266773    4174 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

---
---
---
## Step 2 — Data Collection

### Downloading from Kaggle

There are two ways to download data from Kaggle (works for both numerical and images).
For both you have to sign up and log in.

1. Download `.zip` directly from the Kaggle page
2. Use the Kaggle API

**Method 1 — Kaggle API (recommended):**
```bash
# In your terminal:
pip install kaggle
# Place your kaggle.json API token in ~/.kaggle/
kaggle datasets download -d alessiocorrado99/animals10
unzip animals10.zip -d datasets/
```

**Method 2 — Manual download:**
* Visit https://www.kaggle.com/datasets/alessiocorrado99/animals10/data
* Click **Download** and extract the `.zip` into the `datasets/` folder of your project

After downloading, update the path below if needed.

In [6]:
# ------------------------------------------------------------------
# Update this path to where you extracted the dataset
# ------------------------------------------------------------------
PATH = kagglehub.dataset_download('alessiocorrado99/animals10')
print('Path to dataset files:', path)
# Italian folder names → English class labels
LABEL_MAP = {
    'cane':       'dog',
    'cavallo':    'horse',
    'elefante':   'elephant',
    'farfalla':   'butterfly',
    'gallina':    'chicken',
    'gatto':      'cat',
    'mucca':      'cow',
    'pecora':     'sheep',
    'ragno':      'spider',
    'scoiattolo': 'squirrel'
}

CLASS_NAMES = list(LABEL_MAP.values())  # English class names in folder order
N_LABELS    = len(CLASS_NAMES)
IMG_SIZE    = 64  # Resize all images to 64x64 pixels

print(f"Number of classes: {N_LABELS}")
print(f"Classes: {CLASS_NAMES}")
print(f"Image target size: {IMG_SIZE}x{IMG_SIZE} px")

NameError: name 'kagglehub' is not defined

---
---
---
## Step 3 — Data Validation and Preparation

### 3.1 — Check dataset structure

In [ ]:
# Verify folder structure
folders = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
print(f"Folders found: {folders}")

# Count images per class
image_counts = {}
for folder in folders:
    folder_path = os.path.join(DATASET_PATH, folder)
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    english_name = LABEL_MAP.get(folder, folder)
    image_counts[english_name] = len(files)
    print(f"  {english_name:12s} ({folder}): {len(files)} images")

print(f"\nTotal images: {sum(image_counts.values())}")

### 3.2 — Validate images

Before loading the full dataset, we check that images can be opened and are not corrupted. This follows the validation approach from lesson 8.2.

In [ ]:
def check_images(dataset_path, label_map, sample_size=100):
    """
    Validates a random sample of images from the dataset.
    Checks:
    * File can be opened by PIL
    * Image has 3 colour channels (RGB)
    * Image is not empty / corrupt (no NaN after conversion)
    """
    invalid_count = 0
    valid_count   = 0
    sampled = []

    for folder, english in label_map.items():
        folder_path = os.path.join(dataset_path, folder)
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        sampled.extend([(os.path.join(folder_path, f), english) for f in random.sample(files, min(sample_size, len(files)))])

    for img_path, label in sampled:
        try:
            img = Image.open(img_path).convert('RGB')
            arr = np.array(img)
            if arr.shape[2] != 3:
                print(f"WARNING ({label}): unexpected channels — {img_path}")
                invalid_count += 1
                continue
            if np.isnan(arr).any():
                print(f"WARNING ({label}): NaN values — {img_path}")
                invalid_count += 1
                continue
            valid_count += 1
        except Exception as e:
            print(f"ERROR ({label}): {img_path} — {e}")
            invalid_count += 1

    print(f"\nValidation complete: {valid_count} valid | {invalid_count} invalid (from {len(sampled)} sampled images)")


print("Checking Images...\n")
check_images(DATASET_PATH, LABEL_MAP, sample_size=50)

### 3.3 — Load and resize images into NumPy arrays

All images are resized to `IMG_SIZE × IMG_SIZE` (64×64) and converted to RGB. This is needed because CNN models require a fixed input shape.

In [ ]:
def load_images(dataset_path, label_map, img_size, max_per_class=None):
    """
    Loads images from subfolders, resizes to img_size x img_size,
    converts to RGB NumPy arrays and returns X (images) and y (labels).
    """
    images = []
    labels = []
    class_names_ordered = list(label_map.values())

    for idx, (folder, english) in enumerate(label_map.items()):
        folder_path = os.path.join(dataset_path, folder)
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        if max_per_class is not None:
            files = random.sample(files, min(max_per_class, len(files)))

        loaded = 0
        for fname in files:
            try:
                img = Image.open(os.path.join(folder_path, fname)).convert('RGB')
                img = img.resize((img_size, img_size))
                images.append(np.array(img))
                labels.append(idx)
                loaded += 1
            except Exception:
                pass  # Skip corrupted files

        print(f"  Loaded {loaded} images for '{english}' ({folder})")

    X = np.array(images, dtype='uint8')
    y = np.array(labels, dtype='int32')
    return X, y


print("Loading images...\n")
# max_per_class limits images per class to speed up training.
# Increase or set to None to use the full dataset.
X, y = load_images(DATASET_PATH, LABEL_MAP, img_size=IMG_SIZE, max_per_class=500)

print(f"\nX shape: {X.shape}  |  y shape: {y.shape}")
print(f"Pixel value range: {X.min()} – {X.max()}")

### 3.4 — Split into Train, Validation and Test sets

Following the same approach used in lessons 8.1, 8.2 and 9.1:
* First split: 80% train + 20% test
* Second split: from the train set, reserve 20% as validation

In [ ]:
# First split: train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y  # Maintain class balance
)

# Second split: validation from train
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=0,
    stratify=y_train
)

print("* Train set:     ", X_train.shape, y_train.shape)
print("* Validation set:", X_val.shape,   y_val.shape)
print("* Test set:      ", X_test.shape,  y_test.shape)

### 3.5 — Rescale pixel values and add channel dimension

| Before Scaling (uint8) | After Scaling (float32) |
|---|---|
| Pixel values: 0 – 255 | Pixel values: 0.0 – 1.0 |
| Integer type (uint8) | Floating-point (float32) |
| Can cause instability in training | Helps stable and faster training |

RGB images already have 3 channels, so no reshape is needed.

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_val   = X_val.astype('float32')   / 255.0
X_test  = X_test.astype('float32')  / 255.0

print(f"X_train max after scaling: {X_train.max():.1f}")
print(f"X_train shape: {X_train.shape}  — (samples, height, width, channels)")

In [ ]:
# Convert labels to categorical (one-hot encoding)
y_train_cat = to_categorical(y_train, num_classes=N_LABELS)
y_val_cat   = to_categorical(y_val,   num_classes=N_LABELS)
y_test_cat  = to_categorical(y_test,  num_classes=N_LABELS)

print(f"y_train_cat shape: {y_train_cat.shape}")
print(f"Example one-hot vector: {y_train_cat[0]}")

### 3.6 — Save processed dataset

In [ ]:
os.makedirs('../datasets', exist_ok=True)

np.savez_compressed(
    '../datasets/animals10_processed.npz',
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_train=y_train_cat,
    y_val=y_val_cat,
    y_test=y_test_cat
)

print("Processed dataset saved to '../datasets/animals10_processed.npz'")

---
---
---
## Step 4 — Exploratory Data Analysis (EDA)

### 4.1 — Label frequency distribution

In [ ]:
# Helper function to count labels per dataset split (from lesson 8.2 / 9.1)
df_freq = pd.DataFrame(columns=['Set', 'Label', 'Frequency'])

def count_labels(dataset, dataset_name):
    """
    Helper function to count occurrences of each label and print them.
    """
    global df_freq
    unique, counts = np.unique(dataset, return_counts=True)
    for label, frequency in zip(unique, counts):
        df_freq = pd.concat([
            df_freq,
            pd.DataFrame([{'Set': dataset_name, 'Label': CLASS_NAMES[label], 'Frequency': frequency}])
        ], ignore_index=True)
        print(f"* {dataset_name} - {CLASS_NAMES[label]}: {frequency} images")


count_labels(y_train, 'Train')
print()
count_labels(y_val,   'Validation')
print()
count_labels(y_test,  'Test')

In [ ]:
sns.set_style('whitegrid')
plt.figure(figsize=(12, 6))
sns.barplot(data=df_freq, x='Set', y='Frequency', hue='Label')
plt.title('Label Frequency Distribution in Train, Validation and Test Sets')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 4.2 — Sample images from the dataset

Display one example image per class from the training set.

In [ ]:
sns.set_style('white')
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for class_idx in range(N_LABELS):
    # Find first occurrence of this class in train set
    match_indices = np.where(y_train == class_idx)[0]
    if len(match_indices) > 0:
        sample_img = X_train[match_indices[0]]
        axes[class_idx].imshow(sample_img)
        axes[class_idx].set_title(CLASS_NAMES[class_idx])
        axes[class_idx].axis('off')

plt.suptitle('Sample Images — One Per Class', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3 — Pixel intensity statistics

In [ ]:
# Mean pixel intensity per channel (R, G, B)
channel_names = ['Red', 'Green', 'Blue']
print("Mean pixel intensity per channel (train set, scaled 0–1):")
for i, ch in enumerate(channel_names):
    print(f"  {ch}: {X_train[:, :, :, i].mean():.4f}  |  std: {X_train[:, :, :, i].std():.4f}")

---
---
---
## Step 5 — Building the CNN Model

The architecture follows the pattern from lessons 8.2 and 9.1:
* Two pairs of **Conv2D + MaxPool2D** layers extract features from images
* **Flatten** converts the 2D feature maps into a 1D vector
* **Dense** layers perform the classification
* **Dropout** reduces overfitting
* Output layer uses **softmax** for multi-class probability
* Loss function: **categorical cross-entropy** (standard for multi-class classification)

In [ ]:
def build_tf_model(input_shape, n_labels):
    """
    Builds and compiles a CNN model for image classification.
    Architecture:
      - Two Conv2D + MaxPool2D blocks for feature extraction
      - Flatten + Dense layers for classification
      - Dropout for regularisation
      - Softmax output for multi-class probabilities
    """
    model = Sequential()

    # --- Feature extraction block 1 ---
    model.add(Conv2D(filters=32, kernel_size=(3, 3), input_shape=input_shape, activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    # --- Feature extraction block 2 ---
    model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    # --- Classifier ---
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.25))

    # --- Output layer ---
    model.add(Dense(n_labels, activation='softmax'))

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model


model = build_tf_model(input_shape=X_train.shape[1:], n_labels=N_LABELS)
model.summary()

---
---
---
## Step 6 — Training the Model

**Early stopping** is used to automatically stop training when the validation loss stops improving, preventing overfitting. This follows the approach from lessons 8.1 and 9.1.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    mode='min',
    verbose=1,
    patience=5  # Stop if no improvement after 5 epochs
)

In [ ]:
model = build_tf_model(input_shape=X_train.shape[1:], n_labels=N_LABELS)

model.fit(
    x=X_train,
    y=y_train_cat,
    epochs=30,
    validation_data=(X_val, y_val_cat),
    verbose=1,
    callbacks=[early_stop]
)

### Training history — Loss and Accuracy

In [ ]:
history = pd.DataFrame(model.history.history)
history.head()

In [ ]:
sns.set_style('whitegrid')

history[['loss', 'val_loss']].plot(style='.-', figsize=(10, 4))
plt.title('Training vs. Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Categorical Cross-Entropy Loss')
plt.show()

print('\n')

history[['accuracy', 'val_accuracy']].plot(style='.-', figsize=(10, 4))
plt.title('Training vs. Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.show()

### Save the trained model

In [ ]:
os.makedirs('../models', exist_ok=True)
model.save('../models/animals10_cnn_v1.h5')
print("Model saved to '../models/animals10_cnn_v1.h5'")

---
---
---
## Step 7 — Model Evaluation

### 7.1 — Evaluate on Test Set

In [ ]:
print("Test set evaluation:")
model.evaluate(X_test, y_test_cat)

### 7.2 — Confusion Matrix and Classification Report

Helper functions following the pattern from lesson 9.1.

In [ ]:
def confusion_matrix_and_report(X, y, pipeline, label_map):
    """
    Prints confusion matrix and classification report, and plots a heatmap.
    """
    # Predictions — convert from one-hot to class indices
    prediction = pipeline.predict(X)
    prediction = np.argmax(prediction, axis=1)

    # True labels — convert from one-hot
    y_true = np.argmax(y, axis=1)

    # Compute confusion matrix
    cm = confusion_matrix(y_true=y_true, y_pred=prediction)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        cm,
        columns=['Actual ' + s for s in label_map],
        index=['Predicted ' + s for s in label_map]
    ))
    print('\n')

    print('---  Classification Report  ---')
    print(classification_report(y_true, prediction, target_names=label_map), '\n')

    # Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_map, yticklabels=label_map)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix Heatmap')
    plt.tight_layout()
    plt.show()


def clf_performance(X_train, y_train, X_val, y_val, X_test, y_test, pipeline, label_map):
    """
    Prints classification performance across Train, Validation and Test sets.
    """
    print('#### Train Set ####\n')
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print('#### Validation Set ####\n')
    confusion_matrix_and_report(X_val, y_val, pipeline, label_map)

    print('#### Test Set ####\n')
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

In [ ]:
clf_performance(
    X_train, y_train_cat,
    X_val,   y_val_cat,
    X_test,  y_test_cat,
    model,
    label_map=CLASS_NAMES
)

---
---
---
## Step 8 — Hyperparameter and Architecture Optimisation

We experiment with different model architectures to attempt to improve performance.
A helper function runs each experiment and returns the test accuracy for comparison.

In [ ]:
def experiment(filters_1, filters_2, dense_units, dropout_rate, patience=5, epochs=30):
    """
    Builds and trains a CNN model with the given hyperparameters.
    Returns: test accuracy (float)
    """
    exp_model = Sequential()

    exp_model.add(Conv2D(filters=filters_1, kernel_size=(3, 3),
                         input_shape=X_train.shape[1:], activation='relu'))
    exp_model.add(MaxPool2D(pool_size=(2, 2)))

    exp_model.add(Conv2D(filters=filters_2, kernel_size=(3, 3), activation='relu'))
    exp_model.add(MaxPool2D(pool_size=(2, 2)))

    exp_model.add(Flatten())
    exp_model.add(Dense(dense_units, activation='relu'))
    exp_model.add(Dropout(dropout_rate))

    exp_model.add(Dense(N_LABELS, activation='softmax'))
    exp_model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    es = EarlyStopping(monitor='val_loss', mode='min', verbose=0, patience=patience)
    exp_model.fit(
        x=X_train, y=y_train_cat,
        epochs=epochs,
        validation_data=(X_val, y_val_cat),
        verbose=0,
        callbacks=[es]
    )

    _, test_acc = exp_model.evaluate(X_test, y_test_cat, verbose=0)
    print(f"  filters=({filters_1},{filters_2})  dense={dense_units}  dropout={dropout_rate}  "
          f"→ test accuracy: {test_acc:.4f}")
    return test_acc

In [ ]:
print("Running hyperparameter experiments...\n")

results = []

# Experiment 1 — baseline (matches v1 model above)
acc = experiment(filters_1=32, filters_2=64, dense_units=128, dropout_rate=0.25)
results.append({'filters_1': 32, 'filters_2': 64, 'dense_units': 128, 'dropout_rate': 0.25, 'test_accuracy': acc})

# Experiment 2 — more filters
acc = experiment(filters_1=64, filters_2=128, dense_units=128, dropout_rate=0.25)
results.append({'filters_1': 64, 'filters_2': 128, 'dense_units': 128, 'dropout_rate': 0.25, 'test_accuracy': acc})

# Experiment 3 — higher dropout
acc = experiment(filters_1=32, filters_2=64, dense_units=128, dropout_rate=0.5)
results.append({'filters_1': 32, 'filters_2': 64, 'dense_units': 128, 'dropout_rate': 0.5, 'test_accuracy': acc})

# Experiment 4 — larger dense layer
acc = experiment(filters_1=32, filters_2=64, dense_units=256, dropout_rate=0.25)
results.append({'filters_1': 32, 'filters_2': 64, 'dense_units': 256, 'dropout_rate': 0.25, 'test_accuracy': acc})

print()
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('test_accuracy', ascending=False)
print(results_df.to_string(index=False))

### 8.1 — Train the best model configuration identified above

In [ ]:
# Update these values with the best configuration found from the experiments above
BEST_FILTERS_1   = 64
BEST_FILTERS_2   = 128
BEST_DENSE_UNITS = 128
BEST_DROPOUT     = 0.25

best_model = Sequential()
best_model.add(Conv2D(filters=BEST_FILTERS_1, kernel_size=(3, 3),
                      input_shape=X_train.shape[1:], activation='relu'))
best_model.add(MaxPool2D(pool_size=(2, 2)))

best_model.add(Conv2D(filters=BEST_FILTERS_2, kernel_size=(3, 3), activation='relu'))
best_model.add(MaxPool2D(pool_size=(2, 2)))

best_model.add(Flatten())
best_model.add(Dense(BEST_DENSE_UNITS, activation='relu'))
best_model.add(Dropout(BEST_DROPOUT))
best_model.add(Dense(N_LABELS, activation='softmax'))

best_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

best_model.summary()

In [ ]:
early_stop_best = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5)

best_model.fit(
    x=X_train,
    y=y_train_cat,
    epochs=30,
    validation_data=(X_val, y_val_cat),
    verbose=1,
    callbacks=[early_stop_best]
)

In [ ]:
# Evaluate and plot best model history
best_history = pd.DataFrame(best_model.history.history)

best_history[['loss', 'val_loss']].plot(style='.-', figsize=(10, 4))
plt.title('Best Model — Loss')
plt.show()

best_history[['accuracy', 'val_accuracy']].plot(style='.-', figsize=(10, 4))
plt.title('Best Model — Accuracy')
plt.show()

print("Best model — test set evaluation:")
best_model.evaluate(X_test, y_test_cat)

In [ ]:
clf_performance(
    X_train, y_train_cat,
    X_val,   y_val_cat,
    X_test,  y_test_cat,
    best_model,
    label_map=CLASS_NAMES
)

In [ ]:
best_model.save('../models/animals10_cnn_best.h5')
print("Best model saved to '../models/animals10_cnn_best.h5'")

---
---
---
## Step 9 — Prediction

### 9.1 — Prediction on a random sample from the Test Set

We take a sample from the test set and treat it as if it were live (previously unseen) data, following the approach from lesson 9.1.

In [ ]:
index = 42  # Change to any index to view a different test image

my_animal = X_test[index]
class_index = np.argmax(y_test_cat[index])

print(f"Image shape: {my_animal.shape}")
print(f"True label:  '{CLASS_NAMES[class_index]}'")

sns.set_style('white')
plt.imshow(my_animal)
plt.title(f"True class: {CLASS_NAMES[class_index]}")
plt.axis('off')
plt.show()

In [ ]:
# The model requires a 4D input: (n_samples, height, width, channels)
# We add the batch dimension using np.expand_dims()
live_data = np.expand_dims(my_animal, axis=0)
print(f"Live data shape: {live_data.shape}")

In [ ]:
prediction_proba = best_model.predict(live_data)
prediction_class = np.argmax(prediction_proba, axis=1)[0]

print(f"Predicted class index : {prediction_class}")
print(f"Predicted class name  : '{CLASS_NAMES[prediction_class]}'")
print(f"True class name       : '{CLASS_NAMES[class_index]}'")
print(f"Prediction correct    : {prediction_class == class_index}")

In [ ]:
# Plot prediction probability per class
prob_per_class = pd.DataFrame(data=prediction_proba[0], columns=['Probability'])
prob_per_class = prob_per_class.round(3)
prob_per_class['Results'] = CLASS_NAMES

print(prob_per_class.to_string(index=False))

fig = px.bar(
    prob_per_class,
    x='Results',
    y='Probability',
    range_y=[0, 1],
    width=700, height=400,
    template='seaborn',
    title=f"Prediction probabilities — True class: '{CLASS_NAMES[class_index]}'"
)
fig.update_xaxes(type='category')
fig.show()

### 9.2 — Individual input prediction (from a file path)

A helper function that accepts a path to any image file on disk, preprocesses it the same way as the training data, and returns a prediction.

In [ ]:
def predict_animal(image_path, model, class_names, img_size=64):
    """
    Loads an image from image_path, preprocesses it and predicts the animal class.
    Applies the same preprocessing as the training pipeline:
      - Resize to img_size x img_size
      - Convert to RGB
      - Scale pixel values to [0, 1]
      - Add batch dimension
    Returns: predicted class name and probability DataFrame
    """
    # Load and preprocess
    img = Image.open(image_path).convert('RGB')
    img = img.resize((img_size, img_size))
    img_array = np.array(img).astype('float32') / 255.0

    # Display the image
    sns.set_style('white')
    plt.imshow(img_array)
    plt.title('Input Image')
    plt.axis('off')
    plt.show()

    # Add batch dimension
    live_data = np.expand_dims(img_array, axis=0)

    # Predict
    prediction_proba = model.predict(live_data)
    predicted_idx    = np.argmax(prediction_proba, axis=1)[0]
    predicted_class  = class_names[predicted_idx]

    # Probability table
    prob_df = pd.DataFrame(data=prediction_proba[0], columns=['Probability'])
    prob_df = prob_df.round(3)
    prob_df['Animal'] = class_names

    print(f"Predicted class: '{predicted_class}'")
    print()
    print(prob_df[['Animal', 'Probability']].to_string(index=False))

    # Bar chart
    fig = px.bar(
        prob_df,
        x='Animal',
        y='Probability',
        range_y=[0, 1],
        width=700, height=400,
        template='seaborn',
        title=f"Predicted: '{predicted_class}'"
    )
    fig.update_xaxes(type='category')
    fig.show()

    return predicted_class, prob_df

In [ ]:
# ------------------------------------------------------------------
# Example: Predict from any image file on disk
# Replace the path with your own image.
# ------------------------------------------------------------------

# Use a random test image saved to disk as an example
test_img_path = '../datasets/sample_test_image.jpg'

# Save a test image from our test set for this demonstration
sample_array = (X_test[0] * 255).astype('uint8')
Image.fromarray(sample_array).save(test_img_path)
print(f"Sample image saved to: {test_img_path}")
print(f"True label: {CLASS_NAMES[np.argmax(y_test_cat[0])]}")
print()

# Run individual prediction
predicted_class, prob_df = predict_animal(
    image_path=test_img_path,
    model=best_model,
    class_names=CLASS_NAMES,
    img_size=IMG_SIZE
)

---
---
---
## Summary

This notebook deployed a complete ML Pipeline for an image classification task:

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Data collection (Kaggle Animals10) |
| 3 | Data validation, loading, resizing, splitting, scaling, one-hot encoding |
| 4 | EDA — label distribution, sample images, pixel statistics |
| 5 | CNN model architecture (Conv2D + MaxPool2D + Dense + Dropout) |
| 6 | Model training with EarlyStopping |
| 7 | Evaluation — confusion matrix and classification report |
| 8 | Hyperparameter optimisation via experiment function |
| 9 | Prediction — test set sample and individual input from file |

---
### End of Notebook
##### Reminder: Clear All Outputs before committing.
### Commit and push to GitHub:
```bash
git add --all
git commit -m "Add Animals10 CNN ML pipeline"
git push
```